# Statistics Lab Manual
## Descriptive Statistics using Python (Jupyter Notebook)

**Dataset used:** Adult Census Income Dataset (REAL dataset, ~48,842 rows, from OpenML via `scikit-learn`)

**Level:** 3rd Semester — simple, beginner-friendly Python code

**Topics:**
1. Problem Statement
2. Load Dataset
3. Dependent and Independent Variables
4. Mean vs Median
5. Variance vs IQR
6. Skewness and Distribution Shape
7. Histogram, Boxplot, Density Plot
8. Pivot Tables

---


## 1. Problem Statement

An HR research organization wants to study the **working hours of employed people**
based on the 1994 US Census data.

They want to know:
- What is the average / typical number of working hours per week?
- Does working hours depend on age, education level, or occupation?
- Is the working hours data evenly spread, or are there outliers (people who work
  unusually few or unusually many hours)?

To study this, we will use a **real dataset** called the **Adult Census Income Dataset**
(also known as the "Census Income" dataset). It contains **48,842 rows**, one row per
person, collected from the 1994 US Census Bureau database.

**Dependent variable (what we want to study):** Hours worked per week
**Independent variables (factors that may affect it):** Age, Education, Occupation,
Workclass, Sex, Capital Gain, etc.

## 2. Load Dataset

`scikit-learn` can fetch this real dataset directly from OpenML. We just need to call
one function and it will download and load the data into a pandas DataFrame.
(Internet is required the first time — after that it is cached on your computer.)

In [ ]:
# Step 1: Import the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

print("Libraries imported successfully")

In [ ]:
# Step 2: Load the real Adult Census Income dataset
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame

print("Dataset loaded successfully")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Step 3: Look at the first few rows
df.head()

In [ ]:
# Step 4: Basic information about the dataset
df.info()

In [ ]:
# Step 5: Check for missing values
df.isnull().sum()

In [ ]:
# Step 6: Drop rows with missing values to keep things simple
df = df.dropna()
print("Rows remaining after removing missing values:", df.shape[0])

## 3. Dependent and Independent Variables

| Variable | Type | Meaning |
|---|---|---|
| age | Independent | Age of the person |
| workclass | Independent | Type of employer |
| education | Independent | Highest level of education |
| education-num | Independent | Education level as a number |
| occupation | Independent | Type of job |
| sex | Independent | Gender of the person |
| capital-gain | Independent | Income from investments |
| capital-loss | Independent | Investment losses |
| **hours-per-week** | **Dependent** | **Number of hours worked per week** |

The **dependent variable (Y)** is the one we are trying to explain: `hours-per-week`.
The **independent variables (X)** are the factors that might affect it.

In [ ]:
# Dependent variable
Y = df["hours-per-week"]

# Independent variables
X = df.drop("hours-per-week", axis=1)

print("Dependent variable   :", "hours-per-week")
print("Independent variables:", list(X.columns))

## 4. Mean vs Median Comparison

- **Mean** = sum of values / number of values (average). Gets affected by very large or
  very small values (outliers).
- **Median** = the middle value when data is sorted. Not affected by outliers.

Rule of thumb:
- Mean ≈ Median → data is symmetric.
- Mean > Median → data is right-skewed (some very high values pull the mean up).
- Mean < Median → data is left-skewed (some very low values pull the mean down).

In [ ]:
# Mean and Median of hours-per-week
mean_hours = df["hours-per-week"].mean()
median_hours = df["hours-per-week"].median()

print("Mean of hours-per-week  :", mean_hours)
print("Median of hours-per-week:", median_hours)

if mean_hours > median_hours:
    print("Interpretation: Mean > Median -> data is RIGHT-SKEWED")
elif mean_hours < median_hours:
    print("Interpretation: Mean < Median -> data is LEFT-SKEWED")
else:
    print("Interpretation: Mean = Median -> data is SYMMETRIC")

In [ ]:
# Let's also check Mean vs Median for age
mean_age = df["age"].mean()
median_age = df["age"].median()

print("Mean of age  :", mean_age)
print("Median of age:", median_age)

## 5. Variance vs IQR (Interquartile Range)

- **Variance** and **Standard Deviation** measure how spread out the data is from the
  mean. They use every value, so outliers affect them a lot.
- **IQR = Q3 - Q1** measures the spread of the middle 50% of the data. It is not
  affected much by outliers.

A big difference between what standard deviation suggests and what IQR suggests usually
means the data has outliers.

In [ ]:
# Variance and Standard Deviation
variance_hours = df["hours-per-week"].var()
std_hours = df["hours-per-week"].std()

print("Variance of hours-per-week          :", variance_hours)
print("Standard Deviation of hours-per-week:", std_hours)

In [ ]:
# IQR calculation
Q1 = df["hours-per-week"].quantile(0.25)
Q3 = df["hours-per-week"].quantile(0.75)
IQR = Q3 - Q1

print("Q1 (25th percentile):", Q1)
print("Q3 (75th percentile):", Q3)
print("IQR (Q3 - Q1)       :", IQR)

In [ ]:
# Finding outliers using the 1.5 x IQR rule
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[(df["hours-per-week"] < lower_limit) | (df["hours-per-week"] > upper_limit)]

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Number of outliers found:", len(outliers))

## 6. Skewness and Distribution Shape

**Skewness** tells us if the data is symmetric or leans to one side.

| Skewness value | Meaning |
|---|---|
| close to 0 | Symmetric distribution |
| greater than 0 | Right-skewed (long tail on the right) |
| less than 0 | Left-skewed (long tail on the left) |

In [ ]:
# Skewness of hours-per-week
skewness_hours = df["hours-per-week"].skew()
print("Skewness of hours-per-week:", skewness_hours)

if skewness_hours > 0:
    print("Interpretation: Right-skewed distribution (long tail towards high hours)")
elif skewness_hours < 0:
    print("Interpretation: Left-skewed distribution (long tail towards low hours)")
else:
    print("Interpretation: Symmetric distribution")

In [ ]:
# Skewness of a few more columns
print("Skewness of age          :", df["age"].skew())
print("Skewness of capital-gain :", df["capital-gain"].skew())
print("Skewness of education-num:", df["education-num"].skew())

## 7. Histogram, Boxplot, and Density Plot

- **Histogram** shows how frequently values occur, using bars.
- **Boxplot** shows the median, quartiles, and outliers.
- **Density plot** is a smooth curve version of the histogram.

In [ ]:
# Histogram of hours-per-week
plt.figure(figsize=(8, 5))
plt.hist(df["hours-per-week"], bins=40, color="skyblue", edgecolor="black")
plt.title("Histogram of Hours Worked per Week")
plt.xlabel("Hours per Week")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Boxplot of hours-per-week
plt.figure(figsize=(6, 5))
sns.boxplot(y=df["hours-per-week"], color="lightgreen")
plt.title("Boxplot of Hours Worked per Week")
plt.ylabel("Hours per Week")
plt.show()

In [ ]:
# Density plot of hours-per-week
plt.figure(figsize=(8, 5))
sns.kdeplot(df["hours-per-week"], fill=True, color="orange")
plt.title("Density Plot of Hours Worked per Week")
plt.xlabel("Hours per Week")
plt.show()

**Interpretation:** The histogram shows a very tall spike around 40 hours per week
(the standard full-time work week), with smaller groups working fewer or many more
hours. The boxplot shows outliers on both sides — people working very few hours and
people working very long hours — while the density plot shows this pattern as a sharp
peak with thin tails on both sides.

## 8. Pivot Tables

A pivot table helps us summarize data by groups. Here we will build pivot tables to see
how average working hours change with `sex` and `education`, which are categorical
columns already present in the dataset.

In [ ]:
# Step 1: Check the categories available in 'sex' and 'education'
print("Sex categories:", df["sex"].unique())
print("Education categories:", df["education"].unique())

In [ ]:
# Step 2: Pivot table - Average hours-per-week by Sex
pivot1 = pd.pivot_table(df, values="hours-per-week", index="sex", aggfunc="mean")
pivot1

In [ ]:
# Step 3: Pivot table - Average hours-per-week by Education and Sex
pivot2 = pd.pivot_table(df, values="hours-per-week", index="education",
                         columns="sex", aggfunc="mean")
pivot2

In [ ]:
# Step 4: Visualize the pivot table as a heatmap
plt.figure(figsize=(8, 8))
sns.heatmap(pivot2, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Average Hours Worked per Week by Education and Sex")
plt.show()

**Interpretation:** The pivot table shows that, on average, men tend to work slightly
more hours per week than women across most education levels, and people with higher
education levels (like Doctorate or Prof-school) tend to work more hours per week than
people with lower education levels.

## Lab Summary

In this lab, we used the **real Adult Census Income dataset** to:
1. Understand the problem of studying working hours per week.
2. Load the dataset using `scikit-learn`'s `fetch_openml`.
3. Identify dependent and independent variables.
4. Compare mean and median to detect skew.
5. Compare variance and IQR to detect spread and outliers.
6. Compute skewness to describe the shape of the distribution.
7. Visualize the distribution using histogram, boxplot, and density plot.
8. Build pivot tables to summarize data by groups.

### Practice Exercises
1. Find the mean and median of `age`. Is it symmetric or skewed?
2. Calculate the IQR of `capital-gain` and count the outliers.
3. Make a histogram and boxplot of `education-num`.
4. Create a pivot table showing average `age` by `workclass`.
5. Calculate skewness of `capital-loss` and interpret the result.